# Deep SARSA + UCB–VAE — VN30 (Kaggle, single notebook)

Notebook này tự chứa toàn bộ Environment, ReplayBuffer, VAE, Q-network, Agent và pipeline 291 lượt train. Không import module `.py` từ project.

Pipeline: **VAE screening 27** → **VAE validation 20** → **UCB screening 144** → **Top-5 validation 100**.

> CSV và JSONL được append ngay sau mỗi seed. PNG không có `mode='a'`; biểu đồ được dựng lại từ CSV rồi ghi đè sau mỗi seed để luôn phản ánh checkpoint mới nhất.

In [ ]:
!git clone https://github.com/kohi-vip/SARSA_FinancialRL.git

In [ ]:
!pip install numpy pandas matplotlib tqdm torch TA-Lib optuna

In [ ]:
# BLOCK 1 — Imports, CUDA và cấu hình backbone cố định
from __future__ import annotations

import gc
import json
import math
import os
import random
from collections import deque
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "kaggle_working"
ROOT_RESULTS = WORK_DIR / "deep_sarsa_ucb_vae"
VAE_SCREEN_DIR = ROOT_RESULTS / "vae_screening"
VAE_VALID_DIR = WORK_DIR / "vae_validation_results"
UCB_SCREEN_DIR = ROOT_RESULTS / "ucb_screening"
TOP5_DIR = WORK_DIR / "ucb_top5_validation"
for folder in (ROOT_RESULTS, VAE_SCREEN_DIR, VAE_VALID_DIR, UCB_SCREEN_DIR, TOP5_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATA_CSV = (
    "/kaggle/working/SARSA_FinancialRL/"
    "data/data_storer/data_research/HPG_data.csv"
) # Điền đường dẫn Kaggle nếu auto-discovery không chọn đúng file.
SYMBOL = "HPG"
INITIAL_CASH = 1_000_000_000.0
K = 5
TRANSACTION_COST = 0.001
STATE_DIM = 7
ACTION_DIM = 2 * K + 1

BACKBONE = {
    "episodes": 45,
    "gamma": 0.95,
    "q_lr": 5e-5,
    "hidden_dim": 128,
    "batch_size": 256,
    "replay_capacity": 50_000,
    "warmup_steps": 512,
}
SCREENING_SEEDS = (42, 43, 44)
UCB_SEED_LIST = (1, 2, 3)  # Ba seed co dinh cho moi cau hinh UCB.
VALIDATION_SEEDS = tuple(range(100, 120))
BATCH_IDX = 0  # Tai khoan Kaggle khac doi thu cong thanh 1, 2 hoac 3.
RUN_TOP5_AFTER_BATCH = False  # Chi bat sau khi da gop du 4 batch va chon global Top-5.
EXECUTE_FULL_PIPELINE = False  # Đổi thành True khi đã attach dữ liệu và bật GPU Kaggle.
RESUME = True  # Bỏ qua config_id/seed đã có trong CSV checkpoint.

print("device =", device)
print("output =", WORK_DIR)
print("backbone =", BACKBONE)

## BLOCK 2 — Checkpoint liên tục và dọn CUDA

**BẮT BUỘC SAU MỖI SEED:** append CSV + JSONL, cập nhật JSON snapshot, vẽ lại PNG tiến độ, sau đó `del` các model/buffer lớn, `gc.collect()` và `torch.cuda.empty_cache()`.

In [ ]:
# BLOCK 2 — Reproducibility, append logging, resume và memory cleanup
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def append_seed_checkpoint(csv_path: Path, row: Mapping[str, Any]) -> None:
    """Append một seed vào CSV + JSONL và cập nhật latest JSON ngay lập tức."""
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([json_safe(dict(row))]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False
    )
    jsonl_path = csv_path.with_suffix(".jsonl")
    with jsonl_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(json_safe(dict(row)), ensure_ascii=False) + "\n")
    latest_path = csv_path.with_name(csv_path.stem + "_latest.json")
    latest_path.write_text(json.dumps(json_safe(dict(row)), ensure_ascii=False, indent=2), encoding="utf-8")


def completed_pairs(csv_path: Path) -> set[Tuple[str, int]]:
    if not (RESUME and csv_path.exists()):
        return set()
    frame = pd.read_csv(csv_path)
    if not {"config_id", "seed"}.issubset(frame.columns):
        return set()
    return {(str(row.config_id), int(row.seed)) for row in frame.itertuples()}


def empty_cuda_cache() -> None:
    """Gọi sau khi caller đã `del` model, optimizer, ReplayBuffer và tensor lớn."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def save_progress_scatter(csv_path: Path, png_path: Path, metric: str, title: str) -> None:
    if not csv_path.exists():
        return
    frame = pd.read_csv(csv_path)
    if frame.empty or metric not in frame:
        return
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(np.arange(1, len(frame) + 1), frame[metric], marker="o", ms=3, lw=1)
    ax.set(xlabel="Seed run đã hoàn thành", ylabel=metric, title=title)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(png_path, dpi=180, bbox_inches="tight")  # PNG được cập nhật/ghi đè.
    plt.close(fig)

In [ ]:
# BLOCK 3 — Nạp VNStock CSV, tạo indicator và chia GOOD/BAD
def _normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = [str(c).strip().lower().replace(" ", "_") for c in frame.columns]
    aliases = {
        "date": "time", "datetime": "time", "trading_date": "time",
        "ticker": "symbol", "code": "symbol",
        "adj_close": "close", "close_price": "close",
        "high_price": "high", "low_price": "low",
    }
    return frame.rename(columns={k: v for k, v in aliases.items() if k in frame.columns})


def discover_data_csv() -> Path:
    if DATA_CSV is not None:
        path = Path(DATA_CSV)
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    roots = [Path("/kaggle/input"), Path.cwd() / "data", Path.cwd()]
    candidates: List[Path] = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob("*.csv"))
    candidates = [p for p in candidates if WORK_DIR not in p.parents]
    if not candidates:
        raise FileNotFoundError("Không tìm thấy CSV. Hãy gán DATA_CSV tới file VNStock trên Kaggle.")
    def score(path: Path) -> Tuple[int, int]:
        name = path.name.lower()
        return (int(SYMBOL.lower() in name) * 10 + int("vn30" in name) * 5, -len(str(path)))
    return sorted(candidates, key=score, reverse=True)[0]


def compute_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.sort_values("time").drop_duplicates("time").reset_index(drop=True).copy()
    close = frame["close"].astype(float)
    high = frame.get("high", close).astype(float)
    low = frame.get("low", close).astype(float)
    frame["log_return"] = np.log(close).diff()
    sma20 = close.rolling(20, min_periods=5).mean()
    frame["sma_gap"] = close / sma20 - 1.0
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14, min_periods=5).mean()
    loss = (-delta.clip(upper=0)).rolling(14, min_periods=5).mean()
    rs = gain / loss.replace(0, np.nan)
    frame["rsi_norm"] = ((100 - 100 / (1 + rs)).fillna(50.0) - 50.0) / 50.0
    typical = (high + low + close) / 3.0
    typical_mean = typical.rolling(20, min_periods=5).mean()
    mean_dev = typical.rolling(20, min_periods=5).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
    frame["cci_norm"] = ((typical - typical_mean) / (0.015 * mean_dev.replace(0, np.nan))).clip(-300, 300) / 300.0
    frame["vol20"] = frame["log_return"].rolling(20, min_periods=5).std() * np.sqrt(252)
    columns = ["log_return", "sma_gap", "rsi_norm", "cci_norm", "vol20"]
    frame[columns] = frame[columns].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return frame


def load_and_split_vnstock() -> Dict[str, pd.DataFrame]:
    path = discover_data_csv()
    raw = _normalize_columns(pd.read_csv(path))
    if "symbol" in raw.columns:
        selected = raw[raw["symbol"].astype(str).str.upper() == SYMBOL.upper()]
        if not selected.empty:
            raw = selected
    required = {"time", "close"}
    if not required.issubset(raw.columns):
        raise ValueError(f"CSV phải có {required}; hiện có {list(raw.columns)}")
    raw["time"] = pd.to_datetime(
        raw["time"],
        format="%d/%m/%Y",
        errors="coerce",
    )
    raw["close"] = pd.to_numeric(raw["close"], errors="coerce")
    data = compute_indicators(raw.dropna(subset=["time", "close"]))
    good = data[data["time"] < pd.Timestamp("2022-01-01")].reset_index(drop=True)
    bad = data[(data["time"] >= pd.Timestamp("2022-01-01")) & (data["time"] < pd.Timestamp("2024-01-01"))].reset_index(drop=True)
    if len(good) < 120 or len(bad) < 60:
        raise ValueError(f"Không đủ dữ liệu: GOOD={len(good)}, BAD={len(bad)}")
    split = int(len(good) * 0.8)
    result = {"good_train": good.iloc[:split].reset_index(drop=True),
              "good_valid": good.iloc[split:].reset_index(drop=True),
              "good_full": good, "bad": bad}
    print("data =", path)
    print({key: len(value) for key, value in result.items()})
    return result

In [ ]:
# BLOCK 4 — Environment 7 đặc trưng, action -k..+k, reward danh mục
class VN30TradingEnv:
    FEATURE_COLUMNS = ("log_return", "sma_gap", "rsi_norm", "cci_norm", "vol20")

    def __init__(self, data: pd.DataFrame, initial_cash: float = INITIAL_CASH, k: int = K, fee: float = TRANSACTION_COST):
        if len(data) < 2:
            raise ValueError("Environment cần ít nhất 2 quan sát.")
        self.data = data.reset_index(drop=True)
        self.initial_cash = float(initial_cash)
        self.k = int(k)
        self.fee = float(fee)
        self.actions = np.arange(-self.k, self.k + 1, dtype=np.int64)
        self.reset()

    def reset(self) -> np.ndarray:
        self.index = 0
        self.cash = self.initial_cash
        self.holdings = 0
        self.portfolio_values = [self.initial_cash]
        return self._state()

    def _price(self, index: Optional[int] = None) -> float:
        return float(self.data.iloc[self.index if index is None else index]["close"])

    def _portfolio_value(self, price: Optional[float] = None) -> float:
        price = self._price() if price is None else float(price)
        return float(self.cash + self.holdings * price)

    def _state(self) -> np.ndarray:
        row = self.data.iloc[self.index]
        value = max(self._portfolio_value(), 1e-8)
        dynamic = [self.cash / value, self.holdings * self._price() / value]
        market = [float(row[column]) for column in self.FEATURE_COLUMNS]
        state = np.asarray(dynamic + market, dtype=np.float32)
        assert state.shape == (STATE_DIM,)
        return np.nan_to_num(state, nan=0.0, posinf=0.0, neginf=0.0)

    def valid_action_mask(self) -> np.ndarray:
        price = self._price()
        mask = np.zeros(ACTION_DIM, dtype=bool)
        for idx, quantity in enumerate(self.actions):
            if quantity < 0:
                mask[idx] = self.holdings >= abs(int(quantity))
            elif quantity > 0:
                mask[idx] = self.cash >= quantity * price * (1 + self.fee)
            else:
                mask[idx] = True
        return mask

    def step(self, action_index: int) -> Tuple[np.ndarray, float, bool, Dict[str, float]]:
        action_index = int(action_index)
        if not self.valid_action_mask()[action_index]:
            action_index = self.k  # HOLD nếu action không khả thi.
        quantity = int(self.actions[action_index])
        price = self._price()
        value_before = self._portfolio_value(price)
        if quantity > 0:
            self.cash -= quantity * price * (1 + self.fee)
            self.holdings += quantity
        elif quantity < 0:
            sold = abs(quantity)
            self.cash += sold * price * (1 - self.fee)
            self.holdings -= sold
        self.index += 1
        done = self.index >= len(self.data) - 1
        value_after = self._portfolio_value(self._price())
        reward = (value_after - value_before) / self.initial_cash
        self.portfolio_values.append(value_after)
        return self._state(), float(reward), bool(done), {"portfolio_value": value_after, "quantity": quantity}


In [ ]:
# BLOCK 5 — ReplayBuffer, Q-network 2×128 và Conditional VAE
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=int(capacity))

    def add(self, state, action, reward, next_state, next_action, done) -> None:
        self.buffer.append((np.asarray(state, np.float32), int(action), float(reward),
                            np.asarray(next_state, np.float32), int(next_action), float(done)))

    def __len__(self) -> int:
        return len(self.buffer)

    def sample(self, batch_size: int, target_device: torch.device) -> Tuple[torch.Tensor, ...]:
        batch = random.sample(self.buffer, int(batch_size))
        states, actions, rewards, next_states, next_actions, dones = map(np.asarray, zip(*batch))
        # Mọi batch tensor được đưa thẳng lên GPU/CPU target tại đây.
        return (
            torch.as_tensor(states, dtype=torch.float32, device=target_device),
            torch.as_tensor(actions, dtype=torch.long, device=target_device),
            torch.as_tensor(rewards, dtype=torch.float32, device=target_device),
            torch.as_tensor(next_states, dtype=torch.float32, device=target_device),
            torch.as_tensor(next_actions, dtype=torch.long, device=target_device),
            torch.as_tensor(dones, dtype=torch.float32, device=target_device),
        )


class QNetwork(nn.Module):
    def __init__(self, state_dim: int = STATE_DIM, action_dim: int = ACTION_DIM, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class ConditionalVAE(nn.Module):
    def __init__(self, state_dim: int = STATE_DIM, action_dim: int = ACTION_DIM, latent_dim: int = 16):
        super().__init__()
        input_dim = state_dim + action_dim
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, 128),
                                     nn.ReLU(), nn.Linear(128, input_dim))

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encoder(x)
        return self.mu(hidden), self.logvar(hidden).clamp(-12, 12)

    def forward(self, states: torch.Tensor, action_onehot: torch.Tensor):
        x = torch.cat([states, action_onehot], dim=-1)
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar, x

    @torch.no_grad()
    def novelty(self, states: torch.Tensor, action_onehot: torch.Tensor, delta: float = 1.0) -> torch.Tensor:
        x = torch.cat([states, action_onehot], dim=-1)
        mu, logvar = self.encode(x)
        reconstruction = self.decoder(mu)
        recon = torch.mean((reconstruction - x) ** 2, dim=-1)
        kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)
        return recon + float(delta) * kld


def one_hot_actions(actions: np.ndarray | torch.Tensor, target_device: torch.device = device) -> torch.Tensor:
    tensor = torch.as_tensor(actions, dtype=torch.long, device=target_device)
    return torch.nn.functional.one_hot(tensor, num_classes=ACTION_DIM).float()

In [ ]:
# BLOCK 6 — Dữ liệu VAE, train/evaluate và lưới 9 cấu hình
def collect_random_state_actions(data: pd.DataFrame, trajectories: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    states: List[np.ndarray] = []
    actions: List[int] = []
    for _ in range(int(trajectories)):
        env = VN30TradingEnv(data)
        state = env.reset()
        done = False
        while not done:
            valid = np.flatnonzero(env.valid_action_mask())
            action = int(rng.choice(valid))
            states.append(state)
            actions.append(action)
            state, _, done, _ = env.step(action)
        del env
    return np.asarray(states, np.float32), np.asarray(actions, np.int64)


def vae_loss_terms(model: ConditionalVAE, states: torch.Tensor, actions: torch.Tensor, beta_kl: float):
    reconstruction, mu, logvar, target = model(states, one_hot_actions(actions, states.device))
    recon = torch.mean((reconstruction - target) ** 2)
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + float(beta_kl) * kld, recon, kld


def train_vae_once(train_data: pd.DataFrame, valid_data: pd.DataFrame, config: Mapping[str, Any], seed: int):
    seed_everything(seed)
    train_states, train_actions = collect_random_state_actions(train_data, config.get("trajectories", 5), seed)
    valid_states, valid_actions = collect_random_state_actions(valid_data, 2, seed + 10_000)
    model = ConditionalVAE(latent_dim=int(config["vae_latent_dim"])).to(device)
    optimizer = optim.Adam(model.parameters(), lr=float(config["vae_lr"]))
    batch_size = int(config.get("vae_batch_size", 256))
    history = []
    rng = np.random.default_rng(seed)
    for epoch in range(int(config.get("vae_epochs", 30))):
        order = rng.permutation(len(train_states))
        epoch_loss = epoch_recon = epoch_kld = batches = 0.0
        model.train()
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            states = torch.as_tensor(train_states[indices], dtype=torch.float32, device=device)
            actions = torch.as_tensor(train_actions[indices], dtype=torch.long, device=device)
            loss, recon, kld = vae_loss_terms(model, states, actions, config["vae_beta_kl"])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += float(loss.detach()); epoch_recon += float(recon.detach()); epoch_kld += float(kld.detach()); batches += 1
            del states, actions, loss, recon, kld
        history.append({"loss": epoch_loss / batches, "recon": epoch_recon / batches, "kld": epoch_kld / batches})
    model.eval()
    with torch.no_grad():
        states = torch.as_tensor(valid_states, dtype=torch.float32, device=device)
        actions = torch.as_tensor(valid_actions, dtype=torch.long, device=device)
        val_loss, val_recon, val_kld = vae_loss_terms(model, states, actions, config["vae_beta_kl"])
    metrics = {"train_loss": history[-1]["loss"], "train_recon": history[-1]["recon"],
               "train_kld": history[-1]["kld"], "val_loss": float(val_loss),
               "val_recon": float(val_recon), "val_kld": float(val_kld)}
    del optimizer, states, actions, val_loss, val_recon, val_kld, train_states, train_actions, valid_states, valid_actions
    return model, history, metrics


_vae_lrs = (1e-4, 5e-4, 1e-3)
_vae_betas = (1e-3, 5e-3, 1e-2)
_vae_latents = (8, 16, 32)
VAE_CONFIGS = []
for i, vae_lr in enumerate(_vae_lrs):
    for j, vae_beta_kl in enumerate(_vae_betas):
        VAE_CONFIGS.append({"config_id": f"VAE-{len(VAE_CONFIGS)+1:02d}", "vae_lr": vae_lr,
                            "vae_beta_kl": vae_beta_kl, "vae_latent_dim": _vae_latents[(i + j) % 3],
                            "vae_epochs": 30, "vae_batch_size": 256, "trajectories": 5})
assert len(VAE_CONFIGS) == 9

In [ ]:
# BLOCK 7 — Deep SARSA Agent với bonus UCB từ VAE đóng băng
class UCSarsaAgent:
    def __init__(self, q_network: QNetwork, vae: ConditionalVAE, config: Mapping[str, Any]):
        self.q = q_network
        self.vae = vae
        self.beta = float(config["beta"])
        self.beta_decay = float(config["beta_decay"])
        self.delta = float(config["delta"])

    @torch.no_grad()
    def select_action(self, state: np.ndarray, valid_mask: np.ndarray, episode: int, explore: bool = True) -> int:
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = self.q(state_tensor).squeeze(0)
        scores = q_values.clone()
        if explore:
            states = state_tensor.repeat(ACTION_DIM, 1)
            actions = torch.eye(ACTION_DIM, dtype=torch.float32, device=device)
            novelty = self.vae.novelty(states, actions, delta=self.delta)
            novelty = (novelty - novelty.mean()) / novelty.std().clamp_min(1e-6)
            scores += self.beta * (self.beta_decay ** int(episode)) * novelty
        mask = torch.as_tensor(valid_mask, dtype=torch.bool, device=device)
        scores = scores.masked_fill(~mask, -torch.inf)
        return int(torch.argmax(scores).item())


def optimize_sarsa_batch(q_network: QNetwork, optimizer: optim.Optimizer, replay: ReplayBuffer) -> float:
    states, actions, rewards, next_states, next_actions, dones = replay.sample(BACKBONE["batch_size"], device)
    predicted = q_network(states).gather(1, actions[:, None]).squeeze(1)
    with torch.no_grad():
        next_q = q_network(next_states).gather(1, next_actions[:, None]).squeeze(1)
        target = rewards + BACKBONE["gamma"] * (1.0 - dones) * next_q
    loss = nn.functional.smooth_l1_loss(predicted, target)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
    optimizer.step()
    value = float(loss.detach())
    del states, actions, rewards, next_states, next_actions, dones, predicted, next_q, target, loss
    return value


def portfolio_metrics(values: Sequence[float], dates: Sequence[pd.Timestamp]) -> Dict[str, float]:
    values = np.asarray(values, dtype=np.float64)
    profit = float(values[-1] - values[0])
    returns = np.diff(values) / np.maximum(values[:-1], 1e-12)
    sharpe = float(np.mean(returns) / max(np.std(returns), 1e-12) * np.sqrt(252)) if len(returns) else 0.0
    peaks = np.maximum.accumulate(values)
    max_drawdown = float(abs(np.min((values - peaks) / np.maximum(peaks, 1e-12))) * 100)
    dates = pd.Series(dates).reset_index(drop=True)
    days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = values[-1] / max(values[0], 1e-12)
    arr = float((ratio ** (365.25 / days) - 1) * 100) if ratio > 0 else -100.0
    return {"profit": profit, "arr": arr, "sharpe": sharpe, "max_drawdown": max_drawdown}


def run_ucb_seed(train_data: pd.DataFrame, eval_data: pd.DataFrame, ucb_config: Mapping[str, Any],
                 vae_config: Mapping[str, Any], vae_state_path: Path, seed: int):
    seed_everything(seed)
    vae = ConditionalVAE(latent_dim=int(vae_config["vae_latent_dim"])).to(device)
    vae.load_state_dict(torch.load(vae_state_path, map_location=device, weights_only=True))
    vae.eval()
    for parameter in vae.parameters():
        parameter.requires_grad_(False)
    q_network = QNetwork(hidden_dim=BACKBONE["hidden_dim"]).to(device)
    optimizer = optim.Adam(q_network.parameters(), lr=BACKBONE["q_lr"])
    replay = ReplayBuffer(BACKBONE["replay_capacity"])
    agent = UCSarsaAgent(q_network, vae, ucb_config)
    losses: List[float] = []
    for episode in range(BACKBONE["episodes"]):
        env = VN30TradingEnv(train_data)
        state = env.reset()
        action = agent.select_action(state, env.valid_action_mask(), episode, explore=True)
        done = False
        while not done:
            next_state, reward, done, _ = env.step(action)
            next_action = K if done else agent.select_action(next_state, env.valid_action_mask(), episode, explore=True)
            replay.add(state, action, reward, next_state, next_action, done)
            if len(replay) >= max(BACKBONE["warmup_steps"], BACKBONE["batch_size"]):
                losses.append(optimize_sarsa_batch(q_network, optimizer, replay))
            state, action = next_state, next_action
        del env
    eval_env = VN30TradingEnv(eval_data)
    state = eval_env.reset(); done = False
    while not done:
        action = agent.select_action(state, eval_env.valid_action_mask(), BACKBONE["episodes"], explore=False)
        state, _, done, _ = eval_env.step(action)
    curve = np.asarray(eval_env.portfolio_values, dtype=np.float64)
    metrics = portfolio_metrics(curve, eval_data["time"].iloc[:len(curve)])
    metrics["q_loss"] = float(np.mean(losses[-1000:])) if losses else math.nan
    # **DỌN VRAM:** xóa mọi object giữ tensor CUDA trước khi trả kết quả seed.
    del agent, optimizer, replay, q_network, vae, eval_env, losses
    empty_cuda_cache()
    return metrics, curve

## BLOCK 8 — VAE screening: 9 × 3 = 27 runs

Sau **từng seed**, cell append `vae_screening_metrics.csv`, cập nhật JSONL/JSON/PNG, rồi xóa model và empty CUDA cache.

In [ ]:
# BLOCK 8 — VAE screening và chọn cấu hình tốt nhất
def run_vae_screening(data: Mapping[str, pd.DataFrame]) -> Tuple[Dict[str, Any], pd.DataFrame]:
    csv_path = VAE_SCREEN_DIR / "vae_screening_metrics.csv"
    done = completed_pairs(csv_path)
    expected = {(config["config_id"], seed) for config in VAE_CONFIGS for seed in SCREENING_SEEDS}
    progress = tqdm(total=len(expected), initial=len(done & expected), desc="VAE screening",
                    unit="seed", dynamic_ncols=True)
    for config in VAE_CONFIGS:
        for seed in SCREENING_SEEDS:
            if (config["config_id"], seed) in done:
                continue
            progress.set_postfix(config=config["config_id"], seed=seed)
            model, history, metrics = train_vae_once(data["good_train"], data["good_valid"], config, seed)
            row = {"stage": "vae_screening", "config_id": config["config_id"], "seed": seed,
                   **{k: config[k] for k in ("vae_lr", "vae_latent_dim", "vae_beta_kl")}, **metrics}
            # **SAVE CHECKPOINT SAU MỖI SEED**
            append_seed_checkpoint(csv_path, row)
            save_progress_scatter(csv_path, VAE_SCREEN_DIR / "vae_screening_progress.png",
                                  "val_recon", "VAE screening — validation reconstruction")
            del model, history, metrics
            empty_cuda_cache()  # **EMPTY CUDA CACHE SAU MỖI SEED**
            progress.update(1)
    progress.close()
    frame = pd.read_csv(csv_path)
    summary = frame.groupby("config_id").agg(
        val_recon_mean=("val_recon", "mean"), val_recon_std=("val_recon", "std"),
        val_kld_mean=("val_kld", "mean"), val_loss_mean=("val_loss", "mean"),
    ).reset_index()
    summary["vae_score"] = summary["val_recon_mean"] + summary["val_recon_std"].fillna(0) + 0.1 * summary["val_kld_mean"]
    winner_id = str(summary.sort_values("vae_score").iloc[0]["config_id"])
    winner = next(dict(config) for config in VAE_CONFIGS if config["config_id"] == winner_id)
    summary.to_csv(VAE_SCREEN_DIR / "vae_screening_summary.csv", index=False)
    (VAE_SCREEN_DIR / "best_vae_config.json").write_text(json.dumps(json_safe(winner), indent=2), encoding="utf-8")
    print("Best VAE:", winner)
    return winner, summary

## BLOCK 9 — VAE validation: 1 × 20 = 20 runs

Checkpoint tốt nhất trong 20 seed được lưu làm VAE đóng băng cho UCB. Nhờ vậy không cần train thêm một VAE ngoài ngân sách 291 runs.

In [ ]:
# BLOCK 9 — VAE 20-seed validation, append + distribution plot sau mỗi seed
def save_vae_distribution(csv_path: Path) -> None:
    frame = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, column, title in zip(axes, ("val_loss", "val_recon", "val_kld"),
                                 ("Total loss", "Reconstruction", "KL divergence")):
        ax.boxplot(frame[column].dropna(), showmeans=True)
        ax.scatter(np.ones(len(frame)), frame[column], alpha=0.55, s=18)
        ax.set_title(title); ax.grid(axis="y", alpha=0.25)
    fig.suptitle(f"VAE validation — {len(frame)}/20 seeds")
    fig.tight_layout()
    fig.savefig(VAE_VALID_DIR / "vae_20seeds_distribution.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def run_vae_validation(data: Mapping[str, pd.DataFrame], best_config: Mapping[str, Any]) -> Tuple[Path, pd.DataFrame]:
    csv_path = VAE_VALID_DIR / "vae_20seeds_metrics.csv"
    state_path = VAE_VALID_DIR / "best_frozen_vae_state.pt"
    done = completed_pairs(csv_path)
    best_seen = float(pd.read_csv(csv_path)["val_loss"].min()) if csv_path.exists() else math.inf
    expected = {(best_config["config_id"], seed) for seed in VALIDATION_SEEDS}
    progress = tqdm(total=len(expected), initial=len(done & expected), desc="VAE validation",
                    unit="seed", dynamic_ncols=True)
    for seed in VALIDATION_SEEDS:
        if (best_config["config_id"], seed) in done:
            continue
        progress.set_postfix(config=best_config["config_id"], seed=seed)
        model, history, metrics = train_vae_once(data["good_train"], data["good_valid"], best_config, seed)
        row = {"stage": "vae_validation", "config_id": best_config["config_id"], "seed": seed,
               **{k: best_config[k] for k in ("vae_lr", "vae_latent_dim", "vae_beta_kl")}, **metrics}
        # **SAVE CHECKPOINT SAU MỖI SEED**: CSV append, JSONL append, PNG overwrite.
        append_seed_checkpoint(csv_path, row)
        if metrics["val_loss"] < best_seen:
            best_seen = metrics["val_loss"]
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, state_path)
            (VAE_VALID_DIR / "best_frozen_vae_meta.json").write_text(
                json.dumps(json_safe(row), indent=2), encoding="utf-8")
        save_vae_distribution(csv_path)
        del model, history, metrics
        empty_cuda_cache()  # **EMPTY CUDA CACHE SAU MỖI SEED**
        progress.update(1)
    progress.close()
    if not state_path.exists():
        raise FileNotFoundError("Thiếu frozen VAE checkpoint; hãy chạy lại VAE validation.")
    return state_path, pd.read_csv(csv_path)

In [ ]:
# BLOCK 10 — Chia UCB grid 48 cau hinh thanh 4 batch doc lap.
ALL_UCB_CONFIGS = [
    {"config_id": f"UCB-{index + 1:02d}", "global_index": index,
     "beta": beta, "beta_decay": decay, "delta": delta}
    for index, (beta, decay, delta) in enumerate(
        product((0.03, 0.15, 0.30, 0.85), (0.85, 0.91, 0.94, 0.97), (0.5, 1.0, 2.0)))
]
assert len(ALL_UCB_CONFIGS) == 48
if BATCH_IDX not in range(4):
    raise ValueError(f"BATCH_IDX phai nam trong [0, 1, 2, 3], nhan duoc {BATCH_IDX}")
UCB_BATCH_SIZE = len(ALL_UCB_CONFIGS) // 4
UCB_BATCH_START = BATCH_IDX * UCB_BATCH_SIZE
UCB_BATCH_END = UCB_BATCH_START + UCB_BATCH_SIZE
UCB_CONFIGS = ALL_UCB_CONFIGS[UCB_BATCH_START:UCB_BATCH_END]
assert len(UCB_CONFIGS) == 12
print(f"UCB batch {BATCH_IDX}: global indices {UCB_BATCH_START}..{UCB_BATCH_END - 1}, "
      f"configs={UCB_CONFIGS[0]['config_id']}..{UCB_CONFIGS[-1]['config_id']}")


def append_csv_row(csv_path: Path, row: Mapping[str, Any]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([json_safe(dict(row))]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False
    )


def ucb_config_key(config: Mapping[str, Any]) -> Tuple[float, float, float]:
    return (float(config["beta"]), float(config["beta_decay"]), float(config["delta"]))


def completed_ucb_config_keys(csv_path: Path) -> set[Tuple[float, float, float]]:
    if not (RESUME and csv_path.exists()):
        return set()
    try:
        frame = pd.read_csv(csv_path)
    except (pd.errors.EmptyDataError, OSError):
        return set()
    required = {"beta", "beta_decay", "delta"}
    if not required.issubset(frame.columns):
        return set()
    return {ucb_config_key(row) for row in frame.to_dict(orient="records")}


def rank_ucb(frame: pd.DataFrame) -> pd.DataFrame:
    summary_columns = {"profit_mean", "profit_std", "sharpe_mean", "sharpe_std",
                       "arr_mean", "max_drawdown_mean"}
    if summary_columns.issubset(frame.columns):
        summary = frame.copy()
    else:
        summary = frame.groupby("config_id").agg(
            beta=("beta", "first"), beta_decay=("beta_decay", "first"), delta=("delta", "first"),
            profit_mean=("profit", "mean"), profit_std=("profit", "std"),
            sharpe_mean=("sharpe", "mean"), sharpe_std=("sharpe", "std"),
            arr_mean=("arr", "mean"), max_drawdown_mean=("max_drawdown", "mean"),
        ).reset_index()
    summary["score"] = (summary["sharpe_mean"] - summary["sharpe_std"].fillna(0)
                        + 0.002 * summary["arr_mean"] - 0.01 * summary["max_drawdown_mean"])
    return summary.sort_values("score", ascending=False).reset_index(drop=True)


def run_ucb_screening(data: Mapping[str, pd.DataFrame], best_vae: Mapping[str, Any], vae_state_path: Path) -> pd.DataFrame:
    summary_csv = UCB_SCREEN_DIR / f"ucb_grid_search_batch_{BATCH_IDX}.csv"
    seed_csv = UCB_SCREEN_DIR / f"ucb_grid_search_batch_{BATCH_IDX}_seeds.csv"
    error_csv = UCB_SCREEN_DIR / f"ucb_grid_search_batch_{BATCH_IDX}_errors.csv"
    ranking_csv = UCB_SCREEN_DIR / f"ucb_grid_search_batch_{BATCH_IDX}_ranking.csv"
    completed_configs = completed_ucb_config_keys(summary_csv)
    vae_meta = json.loads((VAE_VALID_DIR / "best_frozen_vae_meta.json").read_text(encoding="utf-8"))

    seed_cache: Dict[Tuple[str, int], Dict[str, Any]] = {}
    if RESUME and seed_csv.exists():
        try:
            for row in pd.read_csv(seed_csv).to_dict(orient="records"):
                seed_cache[(str(row["config_id"]), int(row["seed"]))] = row
        except (pd.errors.EmptyDataError, KeyError, OSError) as error:
            print(f"Ignoring unreadable seed checkpoint {seed_csv}: {error}")

    expected = {(config["config_id"], seed) for config in UCB_CONFIGS for seed in UCB_SEED_LIST}
    already_counted = set(seed_cache) & expected
    for config in UCB_CONFIGS:
        if ucb_config_key(config) in completed_configs:
            already_counted.update((config["config_id"], seed) for seed in UCB_SEED_LIST)
    progress = tqdm(total=len(expected), initial=len(already_counted),
                    desc=f"UCB batch {BATCH_IDX}", unit="seed", dynamic_ncols=True)

    for config in UCB_CONFIGS:
        config_key = ucb_config_key(config)
        if config_key in completed_configs:
            tqdm.write(f"Skipping configuration already completed: {config}")
            continue

        seed_rows: List[Dict[str, Any]] = []
        for seed in UCB_SEED_LIST:
            cache_key = (config["config_id"], int(seed))
            if cache_key in seed_cache:
                seed_rows.append(seed_cache[cache_key])
                continue
            progress.set_postfix(config=config["config_id"], seed=seed)
            try:
                metrics, curve = run_ucb_seed(
                    data["good_train"], data["good_valid"], config, best_vae, vae_state_path, seed
                )
                seed_row = {"stage": "ucb_screening", "batch_idx": BATCH_IDX,
                            "config_id": config["config_id"], "seed": seed, **config, **metrics,
                            "vae_recon": vae_meta["val_recon"], "vae_kld": vae_meta["val_kld"]}
                append_csv_row(seed_csv, seed_row)
                seed_cache[cache_key] = seed_row
                seed_rows.append(seed_row)
                save_progress_scatter(
                    seed_csv, UCB_SCREEN_DIR / f"ucb_batch_{BATCH_IDX}_progress.png",
                    "sharpe", f"UCB batch {BATCH_IDX} - Sharpe by seed"
                )
                del metrics, curve
                progress.update(1)
            except Exception as error:
                error_row = {"timestamp": pd.Timestamp.now().isoformat(), "batch_idx": BATCH_IDX,
                             "config_id": config["config_id"], "seed": seed, **config,
                             "error_type": type(error).__name__, "error": str(error)}
                append_csv_row(error_csv, error_row)
                tqdm.write(f"ERROR configuration={config['config_id']} seed={seed}: {error}")
            finally:
                empty_cuda_cache()

        try:
            unique_seed_rows = {int(row["seed"]): row for row in seed_rows}
            if len(unique_seed_rows) != len(UCB_SEED_LIST):
                tqdm.write(
                    f"Configuration {config['config_id']} incomplete: "
                    f"{len(unique_seed_rows)}/{len(UCB_SEED_LIST)} seeds; it will be retried on resume."
                )
                continue
            ordered_rows = [unique_seed_rows[int(seed)] for seed in UCB_SEED_LIST]
            summary_row: Dict[str, Any] = {
                "batch_idx": BATCH_IDX, "config_id": config["config_id"], **config,
                "seed_list": "|".join(map(str, UCB_SEED_LIST)),
                "seeds_completed": len(ordered_rows),
                "vae_recon": vae_meta["val_recon"], "vae_kld": vae_meta["val_kld"],
            }
            for metric in ("profit", "arr", "sharpe", "max_drawdown"):
                values = np.asarray([float(row[metric]) for row in ordered_rows], dtype=np.float64)
                summary_row[f"{metric}_mean"] = float(values.mean())
                summary_row[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
            append_csv_row(summary_csv, summary_row)
            completed_configs.add(config_key)
            tqdm.write(f"Completed and saved configuration: {config['config_id']}")
        except Exception as error:
            error_row = {"timestamp": pd.Timestamp.now().isoformat(), "batch_idx": BATCH_IDX,
                         "config_id": config["config_id"], "seed": "aggregate", **config,
                         "error_type": type(error).__name__, "error": str(error)}
            append_csv_row(error_csv, error_row)
            tqdm.write(f"ERROR aggregating configuration={config['config_id']}: {error}")
        finally:
            empty_cuda_cache()

    progress.close()
    if not summary_csv.exists():
        print(f"No completed UCB configurations in batch {BATCH_IDX}; see {error_csv}")
        return pd.DataFrame()
    ranking = rank_ucb(pd.read_csv(summary_csv))
    ranking.to_csv(ranking_csv, index=False)
    print(ranking)
    print(f"Batch summary saved to: {summary_csv}")
    return ranking

In [ ]:
# BLOCK 11 — Top-5 × 20 seeds trên BAD, lưu curve và metrics sau từng seed
def save_top5_profit_curves(curve_csv: Path) -> None:
    frame = pd.read_csv(curve_csv)
    fig, ax = plt.subplots(figsize=(12, 5.5))
    for config_id, group in frame.groupby("config_id"):
        mean_curve = group.groupby("step")["portfolio_value"].mean()
        ax.plot(mean_curve.index, mean_curve.values, label=f"{config_id} mean", lw=1.8)
    ax.set(title="Top-5 validation trên BAD — portfolio mean của seed đã hoàn thành",
           xlabel="Trading step", ylabel="Portfolio value")
    ax.grid(alpha=0.25); ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(TOP5_DIR / "top5_profit_curves.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def run_top5_validation(data: Mapping[str, pd.DataFrame], top5: pd.DataFrame,
                        best_vae: Mapping[str, Any], vae_state_path: Path) -> pd.DataFrame:
    metrics_csv = TOP5_DIR / "top5_metrics_progress.csv"
    curves_csv = TOP5_DIR / "top5_portfolio_curves.csv"
    done = completed_pairs(metrics_csv)
    vae_meta = json.loads((VAE_VALID_DIR / "best_frozen_vae_meta.json").read_text(encoding="utf-8"))
    top5_ids = {str(config_id) for config_id in top5["config_id"]}
    expected = {(config_id, seed) for config_id in top5_ids for seed in VALIDATION_SEEDS}
    progress = tqdm(total=len(expected), initial=len(done & expected), desc="Top-5 BAD validation",
                    unit="seed", dynamic_ncols=True)
    for record in top5.to_dict(orient="records"):
        config = {"config_id": str(record["config_id"]), "beta": float(record["beta"]),
                  "beta_decay": float(record["beta_decay"]), "delta": float(record["delta"])}
        for seed in VALIDATION_SEEDS:
            if (config["config_id"], seed) in done:
                continue
            progress.set_postfix(config=config["config_id"], seed=seed)
            metrics, curve = run_ucb_seed(data["good_full"], data["bad"], config, best_vae, vae_state_path, seed)
            row = {"stage": "top5_bad_validation", "config_id": config["config_id"], "seed": seed,
                   **config, **metrics, "vae_recon": vae_meta["val_recon"], "vae_kld": vae_meta["val_kld"]}
            # **SAVE CHECKPOINT SAU MỖI SEED**
            append_seed_checkpoint(metrics_csv, row)
            pd.DataFrame({"config_id": config["config_id"], "seed": seed,
                          "step": np.arange(len(curve)), "portfolio_value": curve}).to_csv(
                curves_csv, mode="a", header=not curves_csv.exists(), index=False
            )
            save_top5_profit_curves(curves_csv)  # PNG cập nhật ngay sau seed.
            del metrics, curve
            empty_cuda_cache()  # **EMPTY CUDA CACHE SAU MỖI SEED**
            progress.update(1)
    progress.close()
    return pd.read_csv(metrics_csv)

In [ ]:
# BLOCK 12 — Final export: Mean ± Std, best_config.json và biểu đồ tổng kết
def final_export(best_vae: Mapping[str, Any], show_plot: bool = True) -> Tuple[Dict[str, Any], pd.DataFrame]:
    metrics_csv = TOP5_DIR / "top5_metrics_progress.csv"
    frame = pd.read_csv(metrics_csv)
    summary = frame.groupby("config_id").agg(
        beta=("beta", "first"), beta_decay=("beta_decay", "first"), delta=("delta", "first"),
        profit_mean=("profit", "mean"), profit_std=("profit", "std"),
        arr_mean=("arr", "mean"), arr_std=("arr", "std"),
        sharpe_mean=("sharpe", "mean"), sharpe_std=("sharpe", "std"),
        max_drawdown_mean=("max_drawdown", "mean"), seeds=("seed", "nunique"),
    ).reset_index()
    summary["robust_score"] = (summary["sharpe_mean"] - summary["sharpe_std"]
                               + 0.002 * summary["arr_mean"] - 0.01 * summary["max_drawdown_mean"])
    summary = summary.sort_values("robust_score", ascending=False).reset_index(drop=True)
    winner = summary.iloc[0].to_dict()
    best = {"vae": dict(best_vae),
            "ucb": {key: winner[key] for key in ("config_id", "beta", "beta_decay", "delta")},
            "backbone": BACKBONE,
            "validation": {key: winner[key] for key in ("profit_mean", "profit_std", "arr_mean",
                                                                  "arr_std", "sharpe_mean", "sharpe_std",
                                                                  "max_drawdown_mean", "seeds")}}
    (TOP5_DIR / "best_config.json").write_text(json.dumps(json_safe(best), ensure_ascii=False, indent=2), encoding="utf-8")
    summary.to_csv(TOP5_DIR / "top5_mean_std_summary.csv", index=False)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    x = np.arange(len(summary)); labels = summary["config_id"]
    for ax, mean_col, std_col, title in zip(axes, ("profit_mean", "arr_mean", "sharpe_mean"),
                                               ("profit_std", "arr_std", "sharpe_std"),
                                               ("Profit mean ± std", "ARR mean ± std", "Sharpe mean ± std")):
        ax.bar(x, summary[mean_col], yerr=summary[std_col], capsize=4)
        ax.set_xticks(x, labels, rotation=35, ha="right"); ax.set_title(title); ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(TOP5_DIR / "top5_mean_std_summary.png", dpi=200, bbox_inches="tight")
    if show_plot:
        plt.show()
    plt.close(fig)
    print(json.dumps(json_safe(best), ensure_ascii=False, indent=2))
    return best, summary

## BLOCK 12.5 — Smoke test toàn bộ pipeline

Đổi `RUN_SMOKE_TEST = True` và chạy riêng cell bên dưới. Test dùng dữ liệu và ngân sách nhỏ, ghi output vào thư mục riêng, sau đó tự khôi phục cấu hình thật. Khi thấy `SMOKE TEST PASSED`, các bước chính đã chạy xuyên suốt không lỗi.

In [ ]:
# Smoke test: 1 VAE config, 1 UCB config, du lieu nho va 1 episode.
RUN_SMOKE_TEST = False


def run_pipeline_smoke_test() -> Dict[str, Any]:
    global SCREENING_SEEDS, UCB_SEED_LIST, VALIDATION_SEEDS, VAE_CONFIGS, UCB_CONFIGS
    global VAE_SCREEN_DIR, VAE_VALID_DIR, UCB_SCREEN_DIR, TOP5_DIR, RESUME

    original_backbone = dict(BACKBONE)
    original_globals = {
        'SCREENING_SEEDS': SCREENING_SEEDS, 'UCB_SEED_LIST': UCB_SEED_LIST,
        'VALIDATION_SEEDS': VALIDATION_SEEDS,
        'VAE_CONFIGS': VAE_CONFIGS, 'UCB_CONFIGS': UCB_CONFIGS,
        'VAE_SCREEN_DIR': VAE_SCREEN_DIR, 'VAE_VALID_DIR': VAE_VALID_DIR,
        'UCB_SCREEN_DIR': UCB_SCREEN_DIR, 'TOP5_DIR': TOP5_DIR, 'RESUME': RESUME,
    }
    stamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S_%f')
    smoke_root = WORK_DIR / 'pipeline_smoke_tests' / stamp

    try:
        VAE_SCREEN_DIR = smoke_root / 'vae_screening'
        VAE_VALID_DIR = smoke_root / 'vae_validation'
        UCB_SCREEN_DIR = smoke_root / 'ucb_screening'
        TOP5_DIR = smoke_root / 'top5_validation'
        for folder in (VAE_SCREEN_DIR, VAE_VALID_DIR, UCB_SCREEN_DIR, TOP5_DIR):
            folder.mkdir(parents=True, exist_ok=True)

        BACKBONE.update({
            'episodes': 1, 'batch_size': 32, 'replay_capacity': 512, 'warmup_steps': 32,
        })
        SCREENING_SEEDS = (42,)
        UCB_SEED_LIST = (42,)
        # Two validation seeds keep standard deviations finite in final_export.
        VALIDATION_SEEDS = (100, 101)
        VAE_CONFIGS = [{
            'config_id': 'VAE-SMOKE', 'vae_lr': 1e-3, 'vae_beta_kl': 1e-2,
            'vae_latent_dim': 8, 'vae_epochs': 1, 'vae_batch_size': 32, 'trajectories': 1,
        }]
        UCB_CONFIGS = [{
            'config_id': 'UCB-SMOKE', 'beta': 0.15, 'beta_decay': 0.94, 'delta': 1.0,
        }]
        RESUME = False

        full_data = load_and_split_vnstock()
        smoke_data = {
            'good_train': full_data['good_train'].iloc[:128].reset_index(drop=True).copy(),
            'good_valid': full_data['good_valid'].iloc[:64].reset_index(drop=True).copy(),
            'good_full': full_data['good_full'].iloc[:160].reset_index(drop=True).copy(),
            'bad': full_data['bad'].iloc[:64].reset_index(drop=True).copy(),
        }
        print('smoke data =', {key: len(value) for key, value in smoke_data.items()})

        best_vae, vae_screen_summary = run_vae_screening(smoke_data)
        vae_state_path, vae_validation = run_vae_validation(smoke_data, best_vae)
        top5 = run_ucb_screening(smoke_data, best_vae, vae_state_path)
        # Run once more with resume enabled; this must skip the completed UCB config.
        RESUME = True
        resumed_ranking = run_ucb_screening(smoke_data, best_vae, vae_state_path)
        if len(resumed_ranking) != len(top5):
            raise RuntimeError('UCB auto-resume smoke check returned inconsistent rows.')
        RESUME = False
        top5_progress = run_top5_validation(smoke_data, top5, best_vae, vae_state_path)
        best_config, final_summary = final_export(best_vae, show_plot=False)

        result = {
            'status': 'passed', 'output_dir': str(smoke_root),
            'best_config': json_safe(best_config),
            'completed': {
                'vae_screening': len(vae_screen_summary),
                'vae_validation': len(vae_validation),
                'ucb_screening': len(top5),
                'top5_validation': len(top5_progress),
                'final_summary': len(final_summary),
            },
        }
        print('SMOKE TEST PASSED:', json.dumps(result, ensure_ascii=False, indent=2))
        return result
    finally:
        BACKBONE.clear()
        BACKBONE.update(original_backbone)
        SCREENING_SEEDS = original_globals['SCREENING_SEEDS']
        UCB_SEED_LIST = original_globals['UCB_SEED_LIST']
        VALIDATION_SEEDS = original_globals['VALIDATION_SEEDS']
        VAE_CONFIGS = original_globals['VAE_CONFIGS']
        UCB_CONFIGS = original_globals['UCB_CONFIGS']
        VAE_SCREEN_DIR = original_globals['VAE_SCREEN_DIR']
        VAE_VALID_DIR = original_globals['VAE_VALID_DIR']
        UCB_SCREEN_DIR = original_globals['UCB_SCREEN_DIR']
        TOP5_DIR = original_globals['TOP5_DIR']
        RESUME = original_globals['RESUME']
        empty_cuda_cache()


if RUN_SMOKE_TEST:
    SMOKE_TEST_RESULTS = run_pipeline_smoke_test()
else:
    print('Smoke test san sang. Doi RUN_SMOKE_TEST=True va chay lai cell nay.')

## BLOCK 13 — Chạy VAE và UCB batch đã chọn

1. Attach CSV VNStock vào Kaggle.
2. Nếu cần, gán `DATA_CSV` ở Block 1.
3. Bật GPU và đổi `EXECUTE_FULL_PIPELINE = True`.
4. Run All. Với `RESUME=True`, notebook bỏ qua seed/cấu hình đã checkpoint.
5. Mặc định `RUN_TOP5_AFTER_BATCH=False`: mỗi tài khoản dừng sau 12 cấu hình. Hãy gộp bốn CSV batch và chọn global Top-5 trước khi validation.

In [ ]:
# BLOCK 13 — Moi tai khoan chay VAE + 12 UCB configs x 3 seeds.
def run_full_research_pipeline() -> Dict[str, Any]:
    data = load_and_split_vnstock()
    best_vae, vae_screen_summary = run_vae_screening(data)
    vae_state_path, vae_validation = run_vae_validation(data, best_vae)
    batch_ranking = run_ucb_screening(data, best_vae, vae_state_path)
    batch_result = {"best_config": None, "vae_screen_summary": vae_screen_summary,
                    "vae_validation": vae_validation, "ucb_batch_ranking": batch_ranking}
    if not RUN_TOP5_AFTER_BATCH:
        print(f"UCB batch {BATCH_IDX} completed. Merge all four batch CSV files before global Top-5 validation.")
        return batch_result
    if batch_ranking.empty:
        raise RuntimeError(f"UCB batch {BATCH_IDX} has no completed configurations.")
    top5 = batch_ranking.head(5).copy()
    top5_progress = run_top5_validation(data, top5, best_vae, vae_state_path)
    best_config, final_summary = final_export(best_vae)
    return {"best_config": best_config, "vae_screen_summary": vae_screen_summary,
            "vae_validation": vae_validation, "top5_screening": top5,
            "top5_progress": top5_progress, "final_summary": final_summary}


if EXECUTE_FULL_PIPELINE:
    PIPELINE_RESULTS = run_full_research_pipeline()
else:
    print("Pipeline đã được định nghĩa. Đổi EXECUTE_FULL_PIPELINE=True ở Block 1 rồi Run All.")